In [1]:
import requests
import pandas as pd
import kaleido
from pathlib import Path
BASE_DIR = Path.cwd()


url_spend = "https://api.worldbank.org/v2/country/all/indicator/SH.XPD.GHED.PP.CD?date=2010:2024&format=json&per_page=20000"
url_life = "https://api.worldbank.org/v2/country/all/indicator/SP.DYN.LE00.IN?date=2010:2024&format=json&per_page=20000"

spend_json = requests.get(url_spend).json()
life_json = requests.get(url_life).json()

spend = pd.DataFrame(spend_json[1])
life = pd.DataFrame(life_json[1])

In [2]:
# Health spending
spend['country_name'] = spend['country'].apply(lambda x: x['value'])
spend = spend[['countryiso3code', 'country_name', 'date', 'value']]
spend = spend.rename(columns={'value': 'health_spending'})

# Life expectancy
life['country_name'] = life['country'].apply(lambda x: x['value'])
life = life[['countryiso3code', 'country_name', 'date', 'value']]
life = life.rename(columns={'value': 'life_expectancy'})

In [3]:
spend = spend[spend['countryiso3code'] != '']
life = life[life['countryiso3code'] != '']

spend['date'] = spend['date'].astype(int)
life['date'] = life['date'].astype(int)

In [4]:
url_meta = "https://api.worldbank.org/v2/country/all?format=json&per_page=20000"
meta_json = requests.get(url_meta).json()
meta = pd.DataFrame(meta_json[1])

In [5]:
meta.head()

,id,iso2Code,name,region,adminregion,incomeLevel,lendingType,capitalCity,longitude,latitude
0,ABW,AW,Aruba,"{'id': 'LCN', 'iso2code': 'ZJ', 'value': 'Lati...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'HIC', 'iso2code': 'XD', 'value': 'High...","{'id': 'LNX', 'iso2code': 'XX', 'value': 'Not ...",Oranjestad,-70.0167,12.5167
1,AFE,ZH,Africa Eastern and Southern,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': 'Aggregates'}",,,
2,AFG,AF,Afghanistan,"{'id': 'MEA', 'iso2code': 'ZQ', 'value': 'Midd...","{'id': 'MNA', 'iso2code': 'XQ', 'value': 'Midd...","{'id': 'LIC', 'iso2code': 'XM', 'value': 'Low ...","{'id': 'IDX', 'iso2code': 'XI', 'value': 'IDA'}",Kabul,69.1761,34.5228
3,AFR,A9,Africa,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': 'Aggregates'}",,,
4,AFW,ZI,Africa Western and Central,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': 'Aggregates'}",,,


In [6]:

meta['countryiso3code'] = meta['id']                                   
meta['income_group'] = meta['incomeLevel'].apply(lambda x: x['value'])  # "High income", etc.
meta['income_id'] = meta['incomeLevel'].apply(lambda x: x['id'])        

# Filtering out aggregates 
meta_countries = meta[meta['income_id'] != 'NA'][['countryiso3code', 'income_group']]

In [7]:
meta_countries.head()


,countryiso3code,income_group
0,ABW,High income
2,AFG,Low income
5,AGO,Lower middle income
6,ALB,Upper middle income
7,AND,High income


In [8]:
df = spend.merge(life, on=['countryiso3code', 'date'], suffixes=('_spend', '_life'))
df = df.merge(meta_countries, on='countryiso3code')

In [9]:
df.head(5)

,countryiso3code,country_name_spend,date,health_spending,country_name_life,life_expectancy,income_group
0,AFG,Afghanistan,2024,NaN,Afghanistan,66.289,Low income
1,AFG,Afghanistan,2023,5.452745,Afghanistan,66.035,Low income
2,AFG,Afghanistan,2022,3.852615,Afghanistan,65.617,Low income
3,AFG,Afghanistan,2021,15.231386,Afghanistan,60.417,Low income
4,AFG,Afghanistan,2020,30.634528,Afghanistan,61.454,Low income


In [10]:

df_clean = df.dropna(subset=['health_spending', 'life_expectancy'])

# the latest year with both values
df_latest = df_clean.sort_values('date').groupby('countryiso3code').last().reset_index()

In [11]:
df_latest['health_spending'] = df_latest['health_spending'].round(1)
df_latest['life_expectancy'] = df_latest['life_expectancy'].round(1)

In [12]:
# What years ended up being "latest" for most countries?
print(df_latest['date'].value_counts())


date
2023    185
2024      7
2021      1
Name: count, dtype: int64


In [13]:
df_latest.head()

,countryiso3code,country_name_spend,date,health_spending,country_name_life,life_expectancy,income_group
0,AFG,Afghanistan,2023,5.5,Afghanistan,66.0,Low income
1,AGO,Angola,2023,110.6,Angola,64.6,Lower middle income
2,ALB,Albania,2023,726.3,Albania,79.6,Upper middle income
3,AND,Andorra,2023,3907.9,Andorra,84.0,High income
4,ARE,United Arab Emirates,2023,2488.9,United Arab Emirates,82.9,High income


In [14]:
# How many countries do we have?
print(df_latest.shape[0])

193


In [15]:
df_latest = df_latest[df_latest['income_group'] != 'Not classified']

In [16]:
print(df_latest['income_group'].unique())

<StringArray>
['Low income', 'Lower middle income', 'Upper middle income', 'High income']
Length: 4, dtype: str


In [17]:
import plotly.express as px
import numpy as np

color_map = {
    'Low income': '#D4D40F',           #color
    'Lower middle income': '#1D981A',  # https://davidmathlogic.com/colorblind/#%23DA3772-%231D981A-%230F6145-%23D4D40F
    'Upper middle income': '#0F6145',  # 
    'High income': '#DA3772'           # 
}

In [18]:

# Fit a log curve (because the relationship flattens at high spending)
df_latest['log_spending'] = np.log(df_latest['health_spending'])
coeffs = np.polyfit(df_latest['log_spending'], df_latest['life_expectancy'], 1)

# Expected life expectancy based on spending
df_latest['expected_le'] = np.polyval(coeffs, df_latest['log_spending'])

# How far below expectation is each country?
df_latest['residual'] = df_latest['life_expectancy'] - df_latest['expected_le']

# Worst underperformer per income group
outliers = df_latest.groupby('income_group').apply(
    lambda g: g.nsmallest(1, 'residual')
).reset_index(drop=True)


outlier_codes = set(outliers['countryiso3code'].values)

fig = px.scatter(
    df_latest,
    x='health_spending',
    y='life_expectancy',
    color='income_group',
    color_discrete_map=color_map,
    hover_name='country_name_spend',
    title='Do Countries That Spend More on Healthcare Live Longer? (2023)',
    labels={
        'health_spending': 'Health Spending per Capita (PPP, $)',
        'life_expectancy': 'Life Expectancy at Birth (years)',
        'income_group': 'Income Group'
    },

)

fig.update_traces(marker=dict(size=6, symbol='circle', line=dict(width=0.5, color='white')))
fig.update_layout(showlegend=True,
    title=dict(
        text='Do Countries That Spend on Healthcare More Live Longer? (2023)<br><sup style="color:gray">Some countries fall below the trend - higher healthcare expenses don\'t add years of life</sup>',
        x=0.5,
        xanchor='center'))

fig.add_annotation(
    text='Source: World Bank Open Data<br>Indicators: SH.XPD.GHED.PP.CD, SP.DYN.LE00.IN',
    xref='paper', yref='paper',
    x=0.98, y=0.02,
    showarrow=False,
    font=dict(size=9, color='gray'),
    align='right'
)


for trace in fig.data:
    income = trace.name
    sizes = []
    for i, row in df_latest[df_latest['income_group'] == income].iterrows():
        if row['countryiso3code'] in outlier_codes:
            sizes.append(14)
        else:
            sizes.append(6)
    trace.marker.size = sizes


fig.show()


In [19]:
fig.write_html("healthcare_chart.html")


In [20]:
fig.to_html(full_html=False, include_plotlyjs='cdn')

'<div>                        <script>window.PlotlyConfig = {MathJaxConfig: \'local\'};</script>\n        <script charset="utf-8" src="https://cdn.plot.ly/plotly-3.4.0.min.js" integrity="sha256-KEmPoupLpFyGMyGAiOsiNDbKDKAvxXAn/W+oQa0ZAfk=" crossorigin="anonymous"></script>                <div id="81a6d295-09c2-4a5e-a063-618e558b9b6e" class="plotly-graph-div" style="height:100%; width:100%;"></div>            <script>                window.PLOTLYENV=window.PLOTLYENV || {};                                if (document.getElementById("81a6d295-09c2-4a5e-a063-618e558b9b6e")) {                    Plotly.newPlot(                        "81a6d295-09c2-4a5e-a063-618e558b9b6e",                        [{"hovertemplate":"\\u003cb\\u003e%{hovertext}\\u003c\\u002fb\\u003e\\u003cbr\\u003e\\u003cbr\\u003eIncome Group=Low income\\u003cbr\\u003eHealth Spending per Capita (PPP, $)=%{x}\\u003cbr\\u003eLife Expectancy at Birth (years)=%{y}\\u003cextra\\u003e\\u003c\\u002fextra\\u003e","hovertext":["Afghani

In [21]:
# Static PNG — for embedding in README
fig.write_image(
    BASE_DIR/"outputs"/"healthcare_chart_2023.png",
    width=1200, height=600, scale=2,
)

# Interactive HTML — for the grader to hover/zoom
fig.write_html(
    BASE_DIR/"outputs"/"healthcare_chart_2023.html"
)

/var/folders/jq/wxn09tld1nlgws0gvk_bcwz00000gn/T/ipykernel_80237/3019723045.py:2: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(
